In [1]:
import os
import math
import json
import warnings
import itertools
from datetime import datetime
from typing import List, Tuple

warnings.filterwarnings("ignore")

In [2]:
os.environ.setdefault("FINTORCH_BASE_TIME_FRAME", "60")

'60'

In [3]:
import numpy as np 
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from fintorch.enum import TimeFrame
from fintorch.exchange import ONLINE_EXCHANGE
from fintorch.utils.preprocess import remove_price_dependency
from fintorch.utils.args import DefaultArgumentParser
from fintorch.utils.plot import draw_candlestick_plot, draw_labels, draw_predictions

In [31]:
string_args = [
    "--start-date", "2024-01-01",
    "--stop-date", "2024-06-01",
    "--time-frame", "60",
    "--symbol", "BTC-USDT"
]

args = DefaultArgumentParser.parse(args=string_args)

In [32]:
sdf = ONLINE_EXCHANGE.future.data.update_candles_dataframe(symbol=args.symbol, time_frame=args.time_frame)
sdf = ONLINE_EXCHANGE.future.data.get_candles_dataframe(symbol=args.symbol, time_frame=args.time_frame).copy()

sdf

,open,high,low,close,volume,trade
timestamp,,,,,,
1705395660,43032.7,43040.0,42985.6,43003.7,201.772,3323
1705395720,43004.5,43016.0,42960.0,42967.3,534.556,4093
1705395780,42967.3,42976.5,42954.2,42957.8,117.148,1847
1705395840,42957.9,42965.7,42928.9,42942.2,245.105,3502
1705395900,42942.3,42953.0,42919.0,42942.6,134.988,1928
...,...,...,...,...,...,...
1712281800,68050.0,68066.0,68022.0,68053.9,39.417,1367
1712281860,68053.9,68095.0,68046.4,68083.5,39.312,1366
1712281920,68083.4,68157.2,68083.4,68154.4,99.288,2164


In [38]:
TRAIN_LENGTH = 480
STEP = 960

window_values = list(range(5, 31, 5))
shift_values = list(range(5, 31, 5))
items = list(itertools.product(window_values, shift_values))

test_profits = []
bar = tqdm(list(range(0, len(sdf), STEP)))
for start_index in bar:
    stop_index = start_index + STEP
    df = sdf[start_index: stop_index]
    df["roc"] = df.close / df.open - 1
    
    best = None
    for window, shift in items:
        df["bmax"] = df.close.rolling(window=window).max().shift(shift)
        df["bmin"] = df.close.rolling(window=window).min().shift(shift)
        df["buy"] = df.bmin < df.open

        train_df = df.iloc[:TRAIN_LENGTH]
        test_df = df.iloc[TRAIN_LENGTH:]
        
        train_profit = round(train_df[train_df.buy].roc.sum(), 4)
        test_profit = round(test_df[test_df.buy].roc.sum(), 4)

        if best is None or best[1] < train_profit:
            best = ((window, shift), train_profit, test_profit)

    test_profits.append(best[2])
    sum_test_profits = round(np.array(test_profits).sum(), 4)
    bar.set_postfix_str(f"sum-test-profits: {sum_test_profits}")

  0%|          | 0/120 [00:00<?, ?it/s]